In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

In [2]:
def ciou_loss(pred, target):

    # Boxes: [x1, y1, x2, y2]
    px1, py1, px2, py2 = pred.unbind(-1)
    tx1, ty1, tx2, ty2 = target.unbind(-1)

    inter_x1 = torch.maximum(px1, tx1)
    inter_y1 = torch.maximum(py1, ty1)
    inter_x2 = torch.minimum(px2, tx2)
    inter_y2 = torch.minimum(py2, ty2)

    inter = (
        torch.clamp(inter_x2 - inter_x1, min=0) *
        torch.clamp(inter_y2 - inter_y1, min=0)
    )

    area_p = (px2-px1) * (py2-py1)
    area_t = (tx2-tx1) * (ty2-ty1)

    union = area_p + area_t - inter
    iou = inter / (union + 1e-7)

    return 1 - iou

In [3]:
def classification_loss(pred_cls, target_cls):

    return F.binary_cross_entropy_with_logits(
        pred_cls,
        target_cls
    )

In [4]:
pred_boxes = torch.tensor([
    [0.1, 0.1, 0.8, 0.8],
    [0.2, 0.2, 0.6, 0.7]
])

target_boxes = torch.tensor([
    [0.15, 0.1, 0.75, 0.8],
    [0.25, 0.2, 0.65, 0.7]
])

pred_cls = torch.randn(2, 1)
target_cls = torch.tensor([
    [1.0],
    [0.0]
])

In [5]:
box_loss = ciou_loss(
    pred_boxes,
    target_boxes
).mean()

cls_loss = classification_loss(
    pred_cls,
    target_cls
)

total_loss = box_loss + cls_loss

print("Box Loss:", box_loss.item())
print("Class Loss:", cls_loss.item())
print("Total Loss:", total_loss.item())

Box Loss: 0.18253985047340393
Class Loss: 0.6492650508880615
Total Loss: 0.8318048715591431


In [6]:
pred_boxes.requires_grad_()
pred_cls.requires_grad_()

box_loss = ciou_loss(
    pred_boxes,
    target_boxes
).mean()

cls_loss = classification_loss(
    pred_cls,
    target_cls
)

loss = box_loss + cls_loss

loss.backward()

print("Box gradients:")
print(pred_boxes.grad)

print("\nClassification gradients:")
print(pred_cls.grad)

Box gradients:
tensor([[-6.1224e-01, -4.3732e-02,  6.1224e-01,  4.3732e-02],
        [-8.6420e-01,  1.1921e-07, -1.1111e+00, -1.1921e-07]])

Classification gradients:
tensor([[-0.2083],
        [ 0.2661]])


In [7]:
print("Anchor-free detection loss completed.")
print("CIoU and classification losses were calculated.")
print("Backpropagation completed successfully.")

Anchor-free detection loss completed.
CIoU and classification losses were calculated.
Backpropagation completed successfully.
